# Evaluating captions before reranking (pre-ranking)

Evaluates the captions produced by the generator with **plain beam search** (beam 3), before the reranker. This is the
"before reranking" point in the paper's tables, and also the first candidate in the notebook
`coco2014-eval-reranking.ipynb`.

During decoding, hallucination is controlled by a **fixed blocking rule** on the logits:
`z'(w) = z(w) − δ · 1[w is a COCO object word, o(w) ∉ E(I)]`, with the evidence set
`E(I) = YOLO-World ∪ {objects named by ≥ r of the K retrieved captions}`. Set `DELTA = 0` to evaluate the model
without blocking (the baseline).

**Configuration** (CONFIG cell): `CHECKPOINT_NAME`, `SPLIT`, `DELTA`. The default is the blocking rule with δ = 3,
YOLO-World `yolov8l-worldv2` at threshold 0.25, K = 5 retrieved captions, r = 3. `CHECKPOINT_NAME` is the epoch-40 checkpoint of
one seed (41, 42, 43): `seed<N>_dynlogits_contrastive_attnpool_epoch40_model.pth` (proposed model) or
`baseline_epoch40_seed<N>.pth` (CE baseline).

**Metrics:** BLEU-1..4, METEOR, ROUGE-L, CIDEr (`simple` tokenisation, as in the paper), CHAIRs, CHAIRi. The CHAIR ground truth
is the union of the objects in the COCO instance labels and the objects named in the 5 reference captions.

**Environment:** the notebook calls the repository's root modules directly (no copies), so the numbers match the scripts
used for the paper. It needs Python with `torch`, `transformers`, `ultralytics`, `pycocoevalcap`, `nltk`, and Java for METEOR
(without Java, METEOR is skipped).

**Running on Kaggle** (T4 or P100 GPU, Internet on). Attach:
- COCO 2014 (`val2014` images, `instances_val2014.json`) and the Karpathy split (`dataset_coco.json`), as in the training notebooks;
- a self-uploaded code + weights dataset with the repository's root modules (`local_captioner.py`, `generate.py`,
  `clipscore.py`, `formula_reranker.py`, `chair_metric.py`, `chair_synonyms.txt`, `yolo_objects.py`, `vocab.py`,
  `schema.py`, `evaluate_reranked_quality.py`, all in one folder), the vocabulary file
  and `yolov8l-worldv2.pt`;
- the checkpoint to evaluate, attached separately (dataset or Kaggle Model) so it is easy to swap, with a file name matching `CHECKPOINT_NAME`. You can also add `train_captions.json` + `train_caption_emb_b32.npy` to skip re-encoding the retrieval corpus.

The notebook finds files by name under `/kaggle/input` and writes results to `/kaggle/working/results/`. Do a test run with
`MAX_IMAGES = 5` first.

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
import json, os, shutil, sys, time
from collections import Counter
from pathlib import Path

# Model to evaluate. CE baseline: dynlogits_contrastive_attnpool_epoch40_model.pth; proposed DL+CL model:
# dynlogits_contrastive_attnpool_epoch40_model.pth. Both share the vocabulary below.
# Give the full path under /kaggle/input or just the file name (the notebook searches every attached input).
CHECKPOINT_NAME = "seed41_dynlogits_contrastive_attnpool_epoch40_model.pth"
VOCAB_NAME = "coco2014_karpathy_restval_dynlogits_contrastive_attnpool_vocab.json"
YOLO_NAME = "yolov8l-worldv2.pt"   # YOLO-World

# The notebook finds files by name in every dataset attached under /kaggle/input, so it does not depend on dataset names.
FILES = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d not in ("train2014", "test2014")]   # skip unused images for speed
    for f in files:
        FILES.setdefault(f, Path(root) / f)


def find(name):
    """Use a full path as is; otherwise search by file name."""
    p = Path(name)
    if p.is_file():
        return p
    assert p.name in FILES, f"{p.name} not found in /kaggle/input, attach the dataset/model that contains it"
    return FILES[p.name]


CODE_DIR = find("local_captioner.py").parent        # code dataset: the repository's root .py modules
KARPATHY_JSON = find("dataset_coco.json")
INSTANCES_JSON = find("instances_val2014.json")
# Karpathy val/test images all belong to val2014; take the folder with the most images to avoid picking a stray sample folder
IMAGE_DIR = Counter(p.parent for n, p in FILES.items() if n.startswith("COCO_val2014_")).most_common(1)[0][0]
# Checkpoint attached separately (dataset or Kaggle Model); to change the model, attach the new one and edit CHECKPOINT_NAME
CHECKPOINT, VOCAB_FILE, YOLO_MODEL = find(CHECKPOINT_NAME), find(VOCAB_NAME), find(YOLO_NAME)
# Pre-encoded retrieval corpus (train_captions.json + train_caption_emb_b32.npy): used if attached, otherwise encoded here
# (only accept a folder that has both files, since some datasets only contain a copy of the .npy file)
RETRIEVAL_CACHE = next((p.parent for p in [FILES.get("train_captions.json")] if p and (p.parent / "train_caption_emb_b32.npy").is_file()),
                       Path("/kaggle/working/retrieval_cache"))
RESULTS_DIR = Path("/kaggle/working/results")

SPLIT = "test"        # "test" | "val"
START, MAX_IMAGES = 0, None   # images sorted by file name as in the paper's reports; e.g. MAX_IMAGES = 5 for a test run
DELTA = 3.0           # strength of the fixed blocking rule; 0 = no blocking

YOLO_CONF, YOLO_MAX_OBJECTS = 0.25, 20   # threshold for the blocking rule and S_det
K_RETR = 5            # number of retrieved captions
MAJ = 3               # majority threshold r: an object named by >= r retrieved captions enters the evidence set
MAX_NEW_TOKENS = 40
FORCE_CPU = False     # True: do not use the GPU

OUTPUT = RESULTS_DIR / f"eval_pre_{CHECKPOINT.stem}_d{DELTA:g}_{SPLIT}.json"

In [ ]:
# ==========================================
# LOAD REPOSITORY CODE, DATA AND CHAIR LABELS
# ==========================================
import subprocess   # requires Internet to be enabled in the notebook settings
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pycocoevalcap", "python-dotenv"], check=True)
if shutil.which("java") is None:   # METEOR needs Java
    subprocess.run("apt-get -qq update && apt-get -qq install -y default-jre-headless > /dev/null", shell=True)
import nltk
for pkg in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    nltk.download(pkg, quiet=True)
USE_METEOR = shutil.which("java") is not None
if not USE_METEOR:
    print("[!] Java not found: skipping METEOR")
if FORCE_CPU:
    import torch
    torch.cuda.is_available = lambda: False   # the repository modules pick their device with this function
sys.path.insert(0, str(CODE_DIR))

import numpy as np
import torch
from tqdm.auto import tqdm

import chair_metric
import local_captioner
import vocab as vocab_mod
from evaluate_reranked_quality import compute_metrics
from PIL import Image
from yolo_objects import YoloObjectExtractor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
data = json.loads(KARPATHY_JSON.read_text(encoding="utf-8"))
items = sorted(({"image": im["filename"], "references": [s["raw"] for s in im["sentences"]]}
                for im in data["images"] if im["split"] == SPLIT), key=lambda r: r["image"])
items = items[START: START + MAX_IMAGES] if MAX_IMAGES else items[START:]
paths = [IMAGE_DIR / r["image"] for r in items]
missing = [p.name for p in paths if not p.is_file()]
assert not missing, f"{len(missing)} images missing, e.g. {missing[:3]}"

OBJECTS, INV = chair_metric.load_synonyms()
fn2id = chair_metric.load_filename_to_cocoid(KARPATHY_JSON)
inst = chair_metric.load_instance_objects_by_cocoid(INSTANCES_JSON, INV)
# CHAIR ground truth = objects in COCO instances  U  objects named in the 5 reference captions
GT = {r["image"]: chair_metric.ground_truth_from_captions(r["references"], OBJECTS, INV) | inst.get(fn2id.get(r["image"]), set())
      for r in items}


class Clip:
    """CLIP ViT-B/32 for the retrieval corpus and S_retr: L2-normalised image/caption vectors, stored as float16."""
    def __init__(self, name="openai/clip-vit-base-patch32"):
        from transformers import CLIPModel, CLIPProcessor
        self.device = DEVICE
        self.model = CLIPModel.from_pretrained(name).to(self.device).eval()
        self.proc = CLIPProcessor.from_pretrained(name)

    @torch.no_grad()
    def _encode(self, fn, batches):
        out = []
        for inp in batches:
            with torch.autocast(self.device, enabled=self.device == "cuda"):
                f = fn(**inp.to(self.device))
            f = f if isinstance(f, torch.Tensor) else f.pooler_output   # transformers >= 5 returns an object; pooler_output is the projected vector (as in clipscore.py)
            out.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu().numpy().astype(np.float16))
        return np.concatenate(out)

    def text(self, texts, bs=512):
        return self._encode(self.model.get_text_features, (self.proc(text=texts[i: i + bs], return_tensors="pt", padding=True,
                                                                     truncation=True) for i in range(0, len(texts), bs)))

    def images(self, paths, bs=64):
        return self._encode(self.model.get_image_features, (self.proc(images=[Image.open(p).convert("RGB") for p in paths[i: i + bs]],
                                                                      return_tensors="pt") for i in range(0, len(paths), bs)))


def minmax(v):
    """Min-max normalisation over the candidates of one image; returns 0.5 when all values are equal."""
    lo, hi = v.min(), v.max()
    return np.full_like(v, 0.5) if hi - lo < 1e-9 else (v - lo) / (hi - lo)


def objs_of(text):
    """List of COCO objects (mapped to class names) mentioned in a caption, as in CHAIR."""
    return chair_metric.caption_to_words(text, OBJECTS, INV)[1]


def score(captions, meteor=USE_METEOR, spice=False):
    """BLEU-1..4, METEOR, ROUGE-L, CIDEr (x100), optional SPICE, CHAIRs/CHAIRi; 'simple' tokenisation as in the paper."""
    refs = [r["references"] for r in items]
    m = compute_metrics(captions, refs, use_meteor=meteor, use_cider=True, use_spice=spice, tokenizer="simple")
    ch = chair_metric.compute_chair({r["image"]: c for r, c in zip(items, captions)}, GT, OBJECTS, INV)
    out = {k: round(float(v) * (100 if k == "CIDEr" else 1), 2) for k, v in m.items()}
    out.update(CHAIRs=round(ch["CHAIRs"], 2), CHAIRi=round(ch["CHAIRi"], 2),
               mean_len=round(float(np.mean([len(c.split()) for c in captions])), 2))
    return out


print(f"{len(items)} {SPLIT} images | device {DEVICE} | checkpoint {CHECKPOINT.name} | δ = {DELTA:g}")

In [ ]:
# ==========================================
# OBJECT EVIDENCE E(I) = YOLO-World  U  {objects named by >= MAJ of the K retrieved captions}
# ==========================================
# Retrieval corpus: all training captions (train + restval), encoded with CLIP ViT-B/32 and cached.
clip32 = Clip()
caps_f, emb_f = RETRIEVAL_CACHE / "train_captions.json", RETRIEVAL_CACHE / "train_caption_emb_b32.npy"
if caps_f.is_file() and emb_f.is_file():
    BANK_CAPS, BANK_EMB = json.loads(caps_f.read_text(encoding="utf-8")), np.load(emb_f)
else:
    BANK_CAPS = [s["raw"].strip().lower() for im in data["images"] if im["split"] in ("train", "restval") for s in im["sentences"]]
    print(f"Encoding the retrieval corpus: {len(BANK_CAPS)} captions (done once)")
    BANK_EMB = clip32.text(BANK_CAPS)
    RETRIEVAL_CACHE.mkdir(parents=True, exist_ok=True)
    caps_f.write_text(json.dumps(BANK_CAPS), encoding="utf-8")
    np.save(emb_f, BANK_EMB)

# K captions closest to the image by image-caption cosine (CLIP B/32); the corpus has no val/test captions, so no label leakage.
img_emb = clip32.images(paths)
dt = torch.float16 if clip32.device == "cuda" else torch.float32
bank_t = torch.from_numpy(BANK_EMB).to(clip32.device, dt)
NB = np.concatenate([(torch.from_numpy(img_emb[i: i + 256]).to(clip32.device, dt) @ bank_t.T).topk(K_RETR, dim=1).indices.cpu().numpy()
                     for i in range(0, len(img_emb), 256)])
del bank_t

extractor = YoloObjectExtractor(str(YOLO_MODEL), YOLO_CONF, YOLO_MAX_OBJECTS)
vocab_mod.set_object_noun_vocab(extractor.class_names())   # object nouns for the negative evidence of S_det
YOLO_OBJS = [extractor.extract(p) for p in tqdm(paths, desc="YOLO-World")]
YOLO_SET = [{INV[o.label] for o in objs if o.confidence >= YOLO_CONF and o.label in INV} for objs in YOLO_OBJS]
NB_COUNTS = [Counter(o for j in idx for o in set(objs_of(BANK_CAPS[j]))) for idx in NB]   # number of retrieved captions naming each object
RETR_SET = [{o for o, n in c.items() if n >= MAJ} for c in NB_COUNTS]   # objects named by >= MAJ retrieved captions
EVIDENCE = [y | r for y, r in zip(YOLO_SET, RETR_SET)]
EVIDENCE_OF = {"union": EVIDENCE, "yolo": YOLO_SET, "retr": RETR_SET}   # evidence source: Y+R, Y only (C8), R only (C9)
print(f"Average {np.mean([len(e) for e in EVIDENCE]):.2f} objects in E(I) per image")

In [ ]:
# ==========================================
# FIXED BLOCKING RULE ON THE LOGITS:  z'(w) = z(w) - δ · 1[w is a COCO object word, o(w) not in E(I)]
# ==========================================
# Applied at every decoding step of both plain beam search and diverse beam search; non-object words are unchanged.
IRREGULAR = {"men": "man", "women": "woman", "kids": "kid", "guys": "guy", "mice": "mouse"}
VOCAB = json.loads(VOCAB_FILE.read_text(encoding="utf-8"))
TOK2OBJ = {}
for word, idx in VOCAB.items():
    w = IRREGULAR.get(word) or chair_metric._LEMMATIZER.lemmatize(word, pos="n")
    if w in OBJECTS:
        TOK2OBJ[idx] = INV[w]
print(f"{len(TOK2OBJ)} vocabulary tokens map to COCO objects")

CURRENT_BIAS = None
if not hasattr(local_captioner, "_ORIG_DECODE_STEP"):
    local_captioner._ORIG_DECODE_STEP = local_captioner.ImageCaptioningTransformer.decode_step


def _biased_decode_step(self, memory, sequence):
    logits = local_captioner._ORIG_DECODE_STEP(self, memory, sequence)
    return logits if CURRENT_BIAS is None else logits + CURRENT_BIAS


local_captioner.ImageCaptioningTransformer.decode_step = _biased_decode_step


def rule_bias(evidence, delta=None):
    delta = DELTA if delta is None else delta
    if not delta:
        return None
    bias = torch.zeros(len(VOCAB))
    for tid, obj in TOK2OBJ.items():
        if obj not in evidence:
            bias[tid] = -delta
    return bias.to(DEVICE)

In [ ]:
# ==========================================
# CAPTION GENERATION: plain beam search (beam 3), with blocking
# ==========================================
captioner = local_captioner.LocalCheckpointCaptioner(checkpoint_path=CHECKPOINT, vocab_path=VOCAB_FILE,
                                                     device=DEVICE, max_length=MAX_NEW_TOKENS)
CAPTIONS = []
t0 = time.time()
for i, path in enumerate(tqdm(paths, desc="Generating captions")):
    CURRENT_BIAS = rule_bias(EVIDENCE[i])
    text, _, _ = captioner.generate_diverse(image_path=path, max_candidates=1, num_beams=3,
                                            num_beam_groups=1, diversity_penalty=0.0)[0]
    CAPTIONS.append(text)
CURRENT_BIAS = None
print(f"{(time.time() - t0) / len(paths):.2f} seconds per image")

In [ ]:
# ==========================================
# SCORING AND SAVING
# ==========================================
METRICS = score(CAPTIONS)
print(json.dumps(METRICS, ensure_ascii=False, indent=1))

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.write_text(json.dumps({
    "checkpoint": str(CHECKPOINT), "vocab": str(VOCAB_FILE), "split": SPLIT, "start": START, "images": len(items),
    "delta": DELTA, "yolo_model": YOLO_MODEL.name, "yolo_conf": YOLO_CONF, "k_retr": K_RETR, "maj": MAJ,
    "metrics": METRICS,
    "rows": [{"image": r["image"], "caption": c, "references": r["references"], "evidence": sorted(e)}
             for r, c, e in zip(items, CAPTIONS, EVIDENCE)],
}, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Saved {OUTPUT}")